# HG4052 · Week 9 Practical
## Your audit table, the worst group, and a TextGrid

**No installs, no model downloads: the transcripts were made in advance. Headphones on.**

The data are real recordings from Mozilla Common Voice: 19 clips by speakers who describe their accent as Singaporean English and 19 by speakers who describe it as United States English, all reading sentences. Each clip was transcribed by the two Week 8 models: **A** = whisper-small, **B** = its Singlish fine-tune.

By the end you will have:
- ✅ a normalizer you completed (the lecture pair falls from 100% to 0% WER)
- ✅ an audit table: WER for both models, overall and by accent, with counts, and one chart
- ✅ one clip that changes the comparison, found by listening
- ✅ three errors from the worst group, listened to and labelled with the A1 categories
- ✅ one boundary checked and one vowel measured in a forced alignment

**How this notebook works.** Click a cell, press **Shift + Enter**, read the output. There is one **✏️ TODO** in code, a single line. Written TODOs: double-click the cell and type.

**Short on time?** Parts 1 to 3 are A1 steps 5 and 6. Part 4 is short.

---
## 0 · Setup

In [ ]:
import os, re, zipfile, difflib, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import wavfile
from IPython.display import Audio, display
warnings.filterwarnings("ignore")

os.makedirs("data", exist_ok=True)
for name in ["class_results.csv", "class_clips.zip", "align_clip.wav", "align_clip.TextGrid"]:
    path = f"data/{name}"
    if not os.path.exists(path):
        try:
            urllib.request.urlretrieve("https://raw.githubusercontent.com/chenchenzi/hg4052-materials/main/week09/" + name, path)
        except Exception as e:
            print("download failed:", e)
    ok = os.path.exists(path) and os.path.getsize(path) > 1000
    print(("✅" if ok else "❌"), f"{name:<22}", (f"{os.path.getsize(path):,} bytes" if ok else "missing (also on NTULearn: download it and upload into data/)"))
if os.path.exists("data/class_clips.zip"):
    zipfile.ZipFile("data/class_clips.zip").extractall("data")
    print("clips unzipped:", len(os.listdir("data/clips")), "files")

---
## 1 · Normalize first

The lecture pair: a correct transcript that scores 100% WER, because of capitals, punctuation, "we'll" and "2". A **normalizer** applies the same fixed rules to both sides before scoring.

**✏️ TODO (one line).** Spell out numerals: look the word up in `NUMBERS`. If you leave it, a finished copy takes over so the rest of the notebook runs.

In [ ]:
NUMBERS = {"0": "zero", "1": "one", "2": "two", "3": "three", "4": "four",
           "5": "five", "6": "six", "7": "seven", "8": "eight", "9": "nine", "10": "ten"}
CONTRACTIONS = {"we'll": "we will", "it's": "it is", "don't": "do not", "i'm": "i am", "can't": "cannot",
                "isn't": "is not", "that's": "that is", "there's": "there is", "won't": "will not", "you're": "you are"}

def normalize(text):
    text = text.lower().replace("’", "'")
    text = re.sub(r"[^\w\s']", " ", text)          # punctuation becomes a space (apostrophes stay)
    words = []
    for w in text.split():
        w = CONTRACTIONS.get(w, w)
        # ✏️ TODO: spell out numerals. Use NUMBERS.get(...): its second argument is what to return
        #          when the word is not in NUMBERS (here: the word itself).
        w = ...
        words.append(w)
    return " ".join(" ".join(words).split())

def normalize_reference(text):
    text = text.lower().replace("’", "'")
    text = re.sub(r"[^\w\s']", " ", text)
    words = [NUMBERS.get(CONTRACTIONS.get(w, w), CONTRACTIONS.get(w, w)) for w in text.split()]
    return " ".join(" ".join(words).split())

try:
    TODO_DONE = normalize("Order 2 kopi.") == "order two kopi"
except TypeError:
    TODO_DONE = False
NORMALIZE = normalize if TODO_DONE else normalize_reference
print("✅ your normalizer works" if TODO_DONE else "⬆ TODO not finished: using normalize_reference for now")

def errors_and_words(reference, hypothesis):
    # Week 5's word edit distance: (substitutions + deletions + insertions, number of reference words)
    r, h = reference.split(), hypothesis.split()
    D = np.zeros((len(r) + 1, len(h) + 1), dtype=int)
    D[:, 0] = np.arange(len(r) + 1); D[0, :] = np.arange(len(h) + 1)
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            D[i, j] = min(D[i-1, j] + 1, D[i, j-1] + 1, D[i-1, j-1] + (r[i-1] != h[j-1]))
    return D[len(r), len(h)], len(r)

REF, HYP = "Okay lah, we’ll order 2 kopi.", "okay lah we will order two kopi"
e, n = errors_and_words(REF, HYP)
print(f"raw:        {e} errors / {n} words = {e/n:.0%}")
e, n = errors_and_words(NORMALIZE(REF), NORMALIZE(HYP))
print(f"normalized: {e} errors / {n} words = {e/n:.0%}   ({NORMALIZE(REF)!r})")

---
## 2 · The audit table (A1 step 5)

One row per clip: file, speaker, accent, gender, reference, and the two hypotheses. For each group, WER is **pooled**: all the errors divided by all the reference words (the lecture's first way to average).

In [ ]:
RESULTS = "data/class_results.csv"      # A1: your own file, with the same columns
GROUP = "accent"                         # A1: the column that holds your contrast

df = pd.read_csv(RESULTS)
for s in ["A", "B"]:
    counts = [errors_and_words(NORMALIZE(r), NORMALIZE(h)) for r, h in zip(df["reference"], df["hyp_" + s].fillna(""))]
    df["err_" + s] = [c[0] for c in counts]
df["words"] = [len(NORMALIZE(r).split()) for r in df["reference"]]

def audit(g):
    return pd.Series({"utterances": len(g), "speakers": g["speaker"].nunique(), "ref words": g["words"].sum(),
                      "WER A": g["err_A"].sum() / g["words"].sum(), "WER B": g["err_B"].sum() / g["words"].sum()})

def show_audit(data):
    table = pd.concat([audit(data).to_frame("all").T, data.groupby(GROUP).apply(audit)])
    display(table.style.format({"WER A": "{:.1%}", "WER B": "{:.1%}", "utterances": "{:.0f}", "speakers": "{:.0f}", "ref words": "{:.0f}"}))
    ax = table[["WER A", "WER B"]].mul(100).plot.bar(rot=0, figsize=(6.5, 3.4), color=["#0e6f66", "#b07c14"])
    ax.set_ylabel("WER (%)"); ax.set_title("WER by system, overall and by " + GROUP)
    plt.tight_layout(); plt.show()

show_audit(df)

**Before you trust the table: listen to the clip with the most errors.**

In [ ]:
top = df.sort_values("err_A", ascending=False).iloc[0]
print(f"{top['file']} ({top['speaker']}, {top[GROUP]}): {top['err_A']} of model A's {df['err_A'].sum()} errors")
print("reference:", top["reference"])
print("model A:  ", top["hyp_A"])
display(Audio(f"data/clips/{top['file']}.mp3"))

**✏️ TODO (in words, double-click).** Is the model wrong, or is the reference wrong for this audio?

> **...**

The next cell sets this clip aside and builds the table again.

In [ ]:
clean = df[df["file"] != top["file"]]
show_audit(clean)

**✏️ TODO (in words, double-click).** With the clip set aside: which accent group has the higher WER for each model, and by how much? What did one clip do to the comparison? Does the Singlish fine-tune (B) do better on Singapore English read speech?

> **...**

---
## 3 · Listen to the errors (A1 step 6)

The cell finds model A's higher-WER group (with the faulty clip set aside) and plays its three clips with the most errors. In the comparison, **[brackets]** mark reference words the model got wrong or dropped, and **{braces}** mark what it wrote instead.

In [ ]:
def compare(ref, hyp):
    r, h = ref.split(), hyp.split(); out = []
    for op, i1, i2, j1, j2 in difflib.SequenceMatcher(a=r, b=h).get_opcodes():
        if op == "equal": out += r[i1:i2]
        else:
            if i2 > i1: out.append("[" + " ".join(r[i1:i2]) + "]")
            if j2 > j1: out.append("{" + " ".join(h[j1:j2]) + "}")
    return " ".join(out)

worst = (clean.groupby(GROUP)["err_A"].sum() / clean.groupby(GROUP)["words"].sum()).idxmax()
print("group with the higher WER for model A:", worst)
for _, row in clean[clean[GROUP] == worst].sort_values("err_A", ascending=False).head(3).iterrows():
    print(f"\n{row['file']} ({row['speaker']}): {row['err_A']} errors from A, {row['err_B']} from B")
    print("  A:", compare(NORMALIZE(row["reference"]), NORMALIZE(row["hyp_A"])))
    print("  B:", compare(NORMALIZE(row["reference"]), NORMALIZE(str(row["hyp_B"]))))
    display(Audio(f"data/clips/{row['file']}.mp3"))

**✏️ TODO (in words, double-click).** Listen to each clip. For three errors, fill in the table with one category from the A1 brief: *phonetically plausible substitution, out-of-vocabulary word or proper name, discourse particle, code-switch or borrowed word, disfluency, normalisation artefact, hallucination, translation*. What in the sound could explain each error?

| clip | said (reference) | written (hypothesis) | category | what in the sound? |
|---|---|---|---|---|
| ... | ... | ... | ... | ... |
| ... | ... | ... | ... | ... |
| ... | ... | ... | ... | ... |

---
## 4 · A forced alignment, checked

The Montreal Forced Aligner placed the word and phone boundaries in one Singapore English clip, given its transcript. Alignments are hypotheses: check them against the spectrogram.

In [ ]:
def read_textgrid(path):
    # minimal reader for a long-format TextGrid: {tier name: [(start, end, label), ...]}
    tiers, name, iv = {}, None, None
    for line in open(path, encoding="utf-8"):
        line = line.strip()
        if line.startswith("name ="): name = line.split("=", 1)[1].strip().strip('"'); tiers[name] = []
        elif line.startswith("xmin =") and name: iv = [float(line.split("=")[1])]
        elif line.startswith("xmax =") and name and iv is not None and len(iv) == 1: iv.append(float(line.split("=")[1]))
        elif line.startswith("text =") and name and iv is not None:
            tiers[name].append((iv[0], iv[1], line.split("=", 1)[1].strip().strip('"'))); iv = None
    return tiers

tg = read_textgrid("data/align_clip.TextGrid")
sr, y = wavfile.read("data/align_clip.wav"); y = y / 32768
t0, t1 = 1.2, 4.95                                   # the speech, without the silences at either end
fig, (a1, a2) = plt.subplots(2, 1, figsize=(13, 5.5), sharex=True, gridspec_kw={"height_ratios": [1, 2]})
tt = np.arange(len(y)) / sr; m = (tt >= t0) & (tt <= t1)
a1.plot(tt[m], y[m], color="0.4", lw=0.5)
a2.specgram(y[(tt >= t0) & (tt <= t1)], NFFT=256, Fs=sr, noverlap=192, cmap="Greys", xextent=(t0, t1))
a2.set_ylim(0, 5000); a2.set_ylabel("Hz"); a2.set_xlabel("time (s)")
for start, end, word in tg["words"]:
    if word and start < t1:
        for ax in (a1, a2): ax.axvline(start, color="#d9472b", lw=1)
        a1.text((start + end) / 2, a1.get_ylim()[1] * 0.85, word, ha="center", fontsize=11, color="#d9472b")
plt.tight_layout(); plt.show()
display(Audio("data/align_clip.wav"))

phones = pd.DataFrame([p for p in tg["phones"] if p[2]], columns=["start", "end", "phone"])
phones["ms"] = ((phones["end"] - phones["start"]) * 1000).round().astype(int)
words = [w for w in tg["words"] if w[2]]
phones["word"] = [next(w[2] for w in words if w[0] <= s < w[1]) for s in phones["start"]]
print(phones.to_string(index=False))

MFA's English dictionary uses a few narrow phone labels; here ɟ and ʎ stand for /g/ and /l/.

**✏️ TODO (in words, double-click).** (1) Pick one word boundary (a red line) and check it on the spectrogram: does it sit where the sound changes? Name one that looks misplaced, or say why they all look right. (2) Find the vowel in *sale* in the table: how long is it in milliseconds?

> **...**

---
## Take-home (optional)

- **Your own A1 file:** in Part 2, set `RESULTS` to your results file (columns: file, speaker, your contrast, reference, hyp_A, hyp_B) and `GROUP` to your contrast column; put your clips in `data/clips/`. Check the data rules in the A1 brief before you upload any corpus audio to Colab.
- **Run an aligner yourself:** the forced-alignment tutorial at chenzixu.rbind.io/resources/1forcedalignment, or Chodroff's tutorial in this week's readings.

---
## ✅ Done looks like

- a normalizer that turns the lecture pair from 100% into 0% WER
- an audit table with counts and a chart, a faulty reference found by listening, and a sentence on the accent gap
- three errors from the worse group, each with a category and a phonetic reason
- one boundary checked and one vowel duration from a machine alignment

**What you built.** A1 steps 5 and 6, rehearsed on real speech: the same table, the same split, the same listening. Next week the direction reverses: from text back to sound.

**Data.** Mozilla Common Voice 17.0, English (CC0); accents are as the speakers described them. Please do not try to identify the speakers.